## ▶ Colab setup — GitHub repo + dataset from Google Drive

Run this cell **first** on Google Colab. It clones the GitHub repo, installs the
`phytolabs` package, mounts your Drive, and unzips + reshapes the Leaf-rust
dataset into `data/{train,val}/{healthy,rust}` so the rest of the notebook runs
on **real data**. Edit `ZIP_PATH` if your zip lives elsewhere in Drive.

This notebook is **standalone**: if the saved GMM / logistic-regression models
from notebooks 01-03 aren't present (a fresh Colab runtime), it trains and saves
them from the data. On a non-Colab machine this cell is a harmless no-op.

In [ ]:
# === Colab setup: GitHub repo + dataset from Google Drive ===================
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Adian17/PhytoLabs.git"
ZIP_PATH = "/content/drive/MyDrive/ece186/WheatLeafRust.zip"  # <-- adjust if needed

if "google.colab" in sys.modules:
    # 1) Clone the repo + install the package.
    if not os.path.isdir("/content/PhytoLabs"):
        !git clone -q $REPO_URL /content/PhytoLabs
    %cd /content/PhytoLabs
    !pip -q install -e .
    # Editable install / _setup aren't importable mid-kernel; add paths explicitly.
    for _p in ("/content/PhytoLabs/src", "/content/PhytoLabs/notebooks"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 2) Mount Drive + unzip the dataset (only the first time).
    if not (Path("data/raw").exists() and any(Path("data/raw").iterdir())):
        from google.colab import drive
        drive.mount("/content/drive")
        !rm -rf data/raw && mkdir -p data/raw
        !unzip -q "$ZIP_PATH" -d data/raw

    # 3) Reshape control/diseased -> data/{train,val}/{healthy,rust} (only once).
    if not (Path("data/train/rust").exists() and any(Path("data/train/rust").glob("*"))):
        raw = Path("data/raw")
        def _find(name):
            cands = [d for d in raw.rglob("*") if d.is_dir() and d.name.lower() == name]
            if not cands:
                raise FileNotFoundError(f"No '{name}' folder under data/raw — check the zip layout.")
            train_cands = [d for d in cands if "train" in str(d).lower()]
            return str((train_cands or cands)[0])
        H, R = _find("control"), _find("diseased")
        print("healthy <-", H, "\nrust    <-", R)
        !python -m scripts.reshape_data --healthy-src "$H" --rust-src "$R" --out data --val-fraction 0.2
    print("dataset:", {c: len(list(Path("data/train", c).glob("*"))) for c in ("healthy", "rust")})

# End-to-end demo

Load the saved GMM + logistic regression and run the full pipeline on the validation set: probability, band label, and a lesion-overlay gallery.

In [ ]:
from _setup import DATA_DIR, ARTIFACTS_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import io, segmentation, pipeline, viz
from phytolabs.logreg import LogisticRegressionSGD

data_dir = ensure_dataset()
gmm_path = ARTIFACTS_DIR / 'gmm.joblib'
logreg_path = ARTIFACTS_DIR / 'logreg.joblib'

# Standalone on Colab: a fresh runtime won't have the artifacts from notebooks
# 01-03, so train + save them here if they're missing.
if gmm_path.exists():
    leaf_gmm = segmentation.LeafGMM.load(gmm_path)
else:
    print('gmm.joblib not found — fitting the GMM...')
    leaf_gmm = pipeline.fit_gmm_from_dir(data_dir / 'train', k=4, limit_per_class=150)
    leaf_gmm.save(gmm_path)

if logreg_path.exists():
    model = LogisticRegressionSGD.load(logreg_path)
else:
    print('logreg.joblib not found — training the classifier...')
    X_tr, y_tr, _ = pipeline.build_feature_table(data_dir / 'train', leaf_gmm)
    model = LogisticRegressionSGD(lr=0.1, epochs=300, batch_size=16, l2=1e-3).fit(X_tr, y_tr)
    model.save(logreg_path)

## Predict on a few validation images

In [ ]:
samples = []
for cls in ('rust', 'healthy'):
    for p in list(io.iter_image_paths(data_dir / 'val' / cls))[:4]:
        result, seg, bgr = pipeline.predict_image(p, leaf_gmm, model)
        caption = f"{cls}: P={result['probability']:.2f} ({result['label']})"
        samples.append((bgr, seg['rust'], caption))
        print(caption, '|', p.name)

## Lesion-overlay gallery (the product UX)

In [ ]:
viz.overlay_gallery(samples, ncols=4)
plt.show()

## Scope reminder

This is **image-level** brown-rust classification. The overlays are a qualitative bonus from the unsupervised GMM; we do **not** claim per-region accuracy because the datasets only provide image-level labels.